In [1]:
from functions import get_model_tokenizer
from sklearn.preprocessing import LabelEncoder
from get_dataset import download_prepare_dataset, dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from sklearn.svm import SVC
from sklearn.metrics import classification_report

/Users/omaratef/miniconda/envs/rbf_fakenews/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model, tokenizer = get_model_tokenizer()

Some weights of the model checkpoint at lanwuwei/GigaBERT-v3-Arabic-and-English were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
le= LabelEncoder()
device = 'mps'
data_df = download_prepare_dataset()
print("Data Distribution: ")
print(data_df['category'].value_counts())
data_df['category'] = le.fit_transform(data_df['category'])
print()
print("Classes are: ", len(le.classes_) , " which are: " ,  le.classes_)

Data Distribution: 
category
Sports      6500
Politics    6500
Religion    6500
Medical     6500
Finance     6500
Tech        6500
Culture     6500
Name: count, dtype: int64

Classes are:  7  which are:  ['Culture' 'Finance' 'Medical' 'Politics' 'Religion' 'Sports' 'Tech']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(data_df["text"], data_df["category"], test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5)
print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%")
print("Validation set is: ",X_val.shape[0], " rows which is ", round(X_val.shape[0]/data_df.shape[0],4)*100, "%")
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%")

Training set is:  31850  rows which is  70.0 %
Validation set is:  6825  rows which is  15.0 %
Testing set is:  6825  rows which is  15.0 %


In [5]:
model.to(device)
with torch.no_grad():
    X_train = model(**tokenizer(X_train.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]
    X_val = model(**tokenizer(X_val.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]
    X_test = model(**tokenizer(X_test.tolist(), max_length = 150, truncation = True, padding = 'max_length', return_tensors = 'pt').to(device)).last_hidden_state[:, 0, :]

RuntimeError: MPS backend out of memory (MPS allocated: 28.35 GB, other allocations: 384.00 KB, max allowed: 36.27 GB). Tried to allocate 13.67 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
X_train = X_train.cpu().numpy()
X_val = X_val.cpu().numpy()
X_test = X_test.cpu().numpy()

In [ ]:
clf = SVC(kernel = 'linear')
clf.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
predictions = clf.predict(X_test)

In [ ]:
print(classification_report(y_test, predictions, target_names=le.classes_))

              precision    recall  f1-score   support

     Culture       0.88      0.78      0.82        18
     Finance       0.96      0.88      0.92        26
     Medical       0.84      0.81      0.82        26
    Politics       1.00      1.00      1.00        22
    Religion       0.90      0.95      0.93        20
      Sports       1.00      1.00      1.00        18
        Tech       0.79      0.95      0.86        20

    accuracy                           0.91       150
   macro avg       0.91      0.91      0.91       150
weighted avg       0.91      0.91      0.91       150

